In [3]:
!pip install xgboost lightgbm prophet plotly scikit-learn pandas numpy

In [4]:
"""
RankOpt Strategic Foresight Platform - NIRF Predictive Engine
A complete ML system for predicting and optimizing university rankings

Author: Strategic Analytics Team
Purpose: Predict NIRF scores and provide strategic insights for rank improvement
"""

# ============================================================================
# PART 1: INSTALLATION AND IMPORTS
# ============================================================================

# Install required packages (run once in Colab)
# !pip install xgboost lightgbm prophet plotly kaleido scikit-learn pandas numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet

# Time series
from statsmodels.tsa.arima.model import ARIMA
from datetime import datetime, timedelta

# ============================================================================
# PART 2: SYNTHETIC DATA GENERATION (For demonstration)
# ============================================================================

def generate_synthetic_nirf_data(n_universities=30, n_years=5):
    """
    Generate synthetic NIRF data for demonstration
    Replace this with your actual data loading function
    """
    np.random.seed(42)

    universities = [f"University_{i}" for i in range(1, n_universities + 1)]
    years = list(range(2019, 2019 + n_years))

    data = []

    for uni in universities:
        base_quality = np.random.uniform(0.5, 0.95)

        for year in years:
            # Simulate metrics with correlations
            publications = int(np.random.normal(150 + base_quality * 300, 50))
            citations = int(publications * np.random.uniform(3, 15) * base_quality)
            patents_granted = int(np.random.poisson(5 * base_quality))
            patents_published = int(np.random.poisson(8 * base_quality))
            projects = int(np.random.normal(20 + base_quality * 30, 10))
            funding = np.random.normal(5 + base_quality * 20, 5) * 1e6  # in lakhs
            phd_faculty = int(np.random.normal(80 + base_quality * 100, 20))
            total_faculty = int(phd_faculty / np.random.uniform(0.7, 0.9))
            students = int(np.random.normal(1500 + base_quality * 2000, 500))

            # Calculate derived metrics
            citations_per_paper = citations / max(publications, 1)
            faculty_student_ratio = total_faculty / max(students, 1)
            phd_faculty_ratio = phd_faculty / max(total_faculty, 1)

            # NIRF Score calculation (simplified model)
            tlr_score = (publications * 0.3 + citations * 0.02 +
                        patents_granted * 5 + patents_published * 3) / 10
            rpc_score = (projects * 2 + funding / 1e6) / 10
            go_score = (faculty_student_ratio * 500 + phd_faculty_ratio * 100) / 10
            oi_score = np.random.uniform(30, 80)
            peer_score = np.random.uniform(40, 90)

            # Weighted NIRF score
            nirf_score = (tlr_score * 0.30 + rpc_score * 0.20 +
                         go_score * 0.15 + oi_score * 0.15 + peer_score * 0.20)

            # Add some noise
            nirf_score += np.random.normal(0, 5)
            nirf_score = max(0, min(100, nirf_score))

            data.append({
                'University': uni,
                'Year': year,
                'Publications_Scopus': publications,
                'Citations_Total': citations,
                'Citations_Per_Paper': citations_per_paper,
                'Patents_Granted': patents_granted,
                'Patents_Published': patents_published,
                'Projects_Count': projects,
                'Funding_Amount_Lakhs': funding / 1e5,
                'PhD_Faculty': phd_faculty,
                'Total_Faculty': total_faculty,
                'PhD_Faculty_Ratio': phd_faculty_ratio,
                'Total_Students': students,
                'Faculty_Student_Ratio': faculty_student_ratio,
                'TLR_Score': tlr_score,
                'RPC_Score': rpc_score,
                'GO_Score': go_score,
                'OI_Score': oi_score,
                'Peer_Score': peer_score,
                'NIRF_Score': nirf_score
            })

    df = pd.DataFrame(data)

    # Calculate ranks within each year
    df['NIRF_Rank'] = df.groupby('Year')['NIRF_Score'].rank(ascending=False, method='min')

    return df

# ============================================================================
# PART 3: DATA PREPROCESSING AND FEATURE ENGINEERING
# ============================================================================

class NIRFDataPreprocessor:
    """Handles data preprocessing and feature engineering"""

    def __init__(self):
        self.scaler = StandardScaler()
        self.feature_columns = None

    def create_features(self, df):
        """Create advanced features from raw data"""
        df = df.copy()

        # Lag features (previous year metrics)
        df = df.sort_values(['University', 'Year'])
        lag_features = ['Publications_Scopus', 'Citations_Total', 'Patents_Granted',
                       'Funding_Amount_Lakhs', 'NIRF_Score']

        for feature in lag_features:
            df[f'{feature}_Lag1'] = df.groupby('University')[feature].shift(1)
            df[f'{feature}_Growth'] = df.groupby('University')[feature].pct_change()

        # Rolling averages
        for feature in lag_features:
            df[f'{feature}_MA2'] = df.groupby('University')[feature].transform(
                lambda x: x.rolling(window=2, min_periods=1).mean()
            )

        # Quality metrics with safe division
        df['Publication_Quality_Index'] = (
            df['Citations_Per_Paper'] * df['Publications_Scopus']
        ) / 1000

        # Avoid division by zero
        df['Research_Intensity'] = df['Funding_Amount_Lakhs'] / (df['Total_Faculty'] + 1e-6)

        df['Innovation_Index'] = (
            (df['Patents_Granted'] * 2 + df['Patents_Published']) /
            (df['Total_Faculty'] + 1e-6)
        )

        # Composite scores
        df['Research_Excellence_Score'] = (
            df['Publications_Scopus'] * 0.4 +
            df['Citations_Total'] * 0.4 +
            df['Patents_Granted'] * 0.2
        ) / 100

        # Replace any inf or -inf values with NaN
        df = df.replace([np.inf, -np.inf], np.nan)

        return df

    def prepare_training_data(self, df, target='NIRF_Score'):
        """Prepare data for model training"""
        # Remove rows with NaN in critical columns
        df = df.dropna(subset=[target])

        # Define feature columns (exclude target, identifiers, and derived scores)
        exclude_cols = ['University', 'Year', 'NIRF_Score', 'NIRF_Rank',
                       'TLR_Score', 'RPC_Score', 'GO_Score', 'OI_Score', 'Peer_Score']

        self.feature_columns = [col for col in df.columns if col not in exclude_cols]

        # Handle any remaining NaN values - calculate median only for numeric columns
        numeric_cols = df[self.feature_columns].select_dtypes(include=[np.number]).columns.tolist()
        df_clean = df[self.feature_columns + [target]].copy()

        # Fill NaN values with median for numeric columns only
        for col in numeric_cols:
            if df_clean[col].isna().any():
                df_clean[col].fillna(df_clean[col].median(), inplace=True)

        # Remove any remaining non-numeric columns from features
        self.feature_columns = [col for col in self.feature_columns if col in numeric_cols]

        # Final safety check: replace any inf/-inf with NaN and then fill
        df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
        for col in self.feature_columns:
            if df_clean[col].isna().any():
                df_clean[col].fillna(df_clean[col].median(), inplace=True)

        # Remove any columns with all NaN or constant values
        self.feature_columns = [col for col in self.feature_columns
                               if df_clean[col].nunique() > 1]

        X = df_clean[self.feature_columns]
        y = df_clean[target]

        return X, y, df_clean

# ============================================================================
# PART 4: ENSEMBLE MODEL IMPLEMENTATION
# ============================================================================

class NIRFEnsembleModel:
    """Ensemble model combining XGBoost, LightGBM, and Random Forest"""

    def __init__(self):
        self.models = {
            'xgboost': xgb.XGBRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                objective='reg:squarederror'
            ),
            'lightgbm': lgb.LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbose=-1
            ),
            'random_forest': RandomForestRegressor(
                n_estimators=200,
                max_depth=10,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42
            )
        }

        self.weights = {'xgboost': 0.4, 'lightgbm': 0.4, 'random_forest': 0.2}
        self.feature_importance = None
        self.training_history = {}

    def train(self, X_train, y_train, X_val=None, y_val=None):
        """Train all models in the ensemble"""
        print("Training Ensemble Models...")
        print("=" * 60)

        for name, model in self.models.items():
            print(f"\nTraining {name}...")

            if name == 'xgboost' and X_val is not None:
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    verbose=False
                )
            elif name == 'lightgbm' and X_val is not None:
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)]
                )
            else:
                model.fit(X_train, y_train)

            # Cross-validation score
            cv_scores = cross_val_score(model, X_train, y_train,
                                       cv=5, scoring='r2')
            print(f"  CV R² Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

            self.training_history[name] = {
                'cv_scores': cv_scores,
                'mean_cv_score': cv_scores.mean()
            }

        # Calculate feature importance
        self._calculate_feature_importance(X_train)

        print("\n" + "=" * 60)
        print("Training Complete!")

    def predict(self, X):
        """Make weighted ensemble predictions"""
        predictions = np.zeros(len(X))

        for name, model in self.models.items():
            pred = model.predict(X)
            predictions += pred * self.weights[name]

        return predictions

    def predict_with_confidence(self, X):
        """Predict with confidence intervals"""
        individual_predictions = []

        for name, model in self.models.items():
            pred = model.predict(X)
            individual_predictions.append(pred)

        predictions = np.average(individual_predictions, axis=0,
                                weights=list(self.weights.values()))

        # Calculate standard deviation across models as uncertainty measure
        std = np.std(individual_predictions, axis=0)

        # 95% confidence interval (approximately 2 standard deviations)
        lower_bound = predictions - 2 * std
        upper_bound = predictions + 2 * std

        return predictions, lower_bound, upper_bound

    def _calculate_feature_importance(self, X_train):
        """Calculate and aggregate feature importance"""
        importance_dict = {}

        for name, model in self.models.items():
            if hasattr(model, 'feature_importances_'):
                importance_dict[name] = model.feature_importances_

        # Weighted average of importances
        self.feature_importance = pd.DataFrame({
            'Feature': X_train.columns,
            'Importance': np.average(
                list(importance_dict.values()),
                axis=0,
                weights=[self.weights[k] for k in importance_dict.keys()]
            )
        }).sort_values('Importance', ascending=False)

# ============================================================================
# PART 5: TIME SERIES FORECASTING
# ============================================================================

class TimeSeriesForecaster:
    """Forecast individual metrics using ARIMA and Prophet"""

    def __init__(self):
        self.arima_models = {}
        self.prophet_models = {}

    def forecast_metric_arima(self, df, university, metric, periods=3):
        """Forecast a single metric using ARIMA"""
        uni_data = df[df['University'] == university].sort_values('Year')

        if len(uni_data) < 3:
            return None

        values = uni_data[metric].values

        try:
            # Fit ARIMA model
            model = ARIMA(values, order=(1, 1, 1))
            fitted_model = model.fit()

            # Forecast
            forecast = fitted_model.forecast(steps=periods)

            return forecast
        except:
            # If ARIMA fails, use simple linear extrapolation
            years = uni_data['Year'].values
            slope = (values[-1] - values[0]) / (years[-1] - years[0])
            forecast = [values[-1] + slope * (i + 1) for i in range(periods)]
            return np.array(forecast)

    def forecast_all_metrics(self, df, university, metrics, periods=3):
        """Forecast multiple metrics for a university"""
        forecasts = {}

        for metric in metrics:
            forecast = self.forecast_metric_arima(df, university, metric, periods)
            if forecast is not None:
                forecasts[metric] = forecast

        return forecasts

# ============================================================================
# PART 6: STRATEGIC DASHBOARD AND VISUALIZATION
# ============================================================================

class StrategicDashboard:
    """Generate interactive visualizations and insights"""

    def __init__(self, model, preprocessor, data):
        self.model = model
        self.preprocessor = preprocessor
        self.data = data

    def plot_feature_importance(self, top_n=15):
        """Plot top feature importances"""
        fig = px.bar(
            self.model.feature_importance.head(top_n),
            x='Importance',
            y='Feature',
            orientation='h',
            title=f'Top {top_n} Most Important Features for NIRF Score Prediction',
            labels={'Importance': 'Feature Importance', 'Feature': ''},
            color='Importance',
            color_continuous_scale='Viridis'
        )

        fig.update_layout(height=600, showlegend=False)
        fig.show()

    def plot_prediction_accuracy(self, y_true, y_pred):
        """Plot actual vs predicted scores"""
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=y_true,
            y=y_pred,
            mode='markers',
            name='Predictions',
            marker=dict(size=8, color='blue', opacity=0.6)
        ))

        # Perfect prediction line
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode='lines',
            name='Perfect Prediction',
            line=dict(color='red', dash='dash')
        ))

        fig.update_layout(
            title='Actual vs Predicted NIRF Scores',
            xaxis_title='Actual NIRF Score',
            yaxis_title='Predicted NIRF Score',
            height=500
        )

        fig.show()

    def plot_rank_trajectory(self, university, forecasts, current_year=2023):
        """Plot historical and projected rank trajectory"""
        uni_data = self.data[self.data['University'] == university].sort_values('Year')

        years = uni_data['Year'].tolist()
        ranks = uni_data['NIRF_Rank'].tolist()

        # Add forecasted years
        forecast_years = [current_year + i + 1 for i in range(len(forecasts))]

        fig = go.Figure()

        # Historical data
        fig.add_trace(go.Scatter(
            x=years,
            y=ranks,
            mode='lines+markers',
            name='Historical Rank',
            line=dict(color='blue', width=3),
            marker=dict(size=10)
        ))

        # Forecasted data
        fig.add_trace(go.Scatter(
            x=forecast_years,
            y=forecasts,
            mode='lines+markers',
            name='Projected Rank',
            line=dict(color='orange', width=3, dash='dash'),
            marker=dict(size=10, symbol='diamond')
        ))

        fig.update_layout(
            title=f'Rank Trajectory for {university}',
            xaxis_title='Year',
            yaxis_title='NIRF Rank',
            yaxis=dict(autorange='reversed'),  # Lower rank is better
            height=500,
            hovermode='x unified'
        )

        fig.show()

    def generate_strategic_recommendations(self, university, current_metrics, target_rank=30):
        """Generate actionable recommendations based on feature importance"""
        top_features = self.model.feature_importance.head(10)

        recommendations = []

        print(f"\n{'='*70}")
        print(f"STRATEGIC RECOMMENDATIONS FOR {university}")
        print(f"Target Rank: {target_rank}")
        print(f"{'='*70}\n")

        for idx, row in top_features.iterrows():
            feature = row['Feature']
            importance = row['Importance']

            if 'Publication' in feature:
                rec = f"📚 Increase research publications (Importance: {importance:.3f})"
                action = "   → Incentivize faculty publications in Q1/Q2 journals"
                recommendations.append((rec, action))

            elif 'Citation' in feature:
                rec = f"📊 Improve citation impact (Importance: {importance:.3f})"
                action = "   → Focus on high-impact research areas and collaborations"
                recommendations.append((rec, action))

            elif 'Patent' in feature:
                rec = f"💡 Strengthen IPR portfolio (Importance: {importance:.3f})"
                action = "   → Establish innovation cell and patent filing support"
                recommendations.append((rec, action))

            elif 'Funding' in feature:
                rec = f"💰 Increase research funding (Importance: {importance:.3f})"
                action = "   → Apply for more government and industry grants"
                recommendations.append((rec, action))

            elif 'Faculty' in feature:
                rec = f"👨‍🏫 Enhance faculty quality (Importance: {importance:.3f})"
                action = "   → Recruit PhD-qualified faculty and improve ratio"
                recommendations.append((rec, action))

        for i, (rec, action) in enumerate(recommendations[:5], 1):
            print(f"{i}. {rec}")
            print(action)
            print()

        print(f"{'='*70}\n")

# ============================================================================
# PART 7: MAIN EXECUTION PIPELINE
# ============================================================================

def main_pipeline():
    """Main execution pipeline for the RankOpt system"""

    print("="*70)
    print("RANKOPT STRATEGIC FORESIGHT PLATFORM")
    print("NIRF Predictive Engine v1.0")
    print("="*70)
    print()

    # Step 1: Load or generate data
    print("Step 1: Loading data...")
    df = generate_synthetic_nirf_data(n_universities=30, n_years=5)
    print(f"  Loaded data: {len(df)} records, {df['University'].nunique()} universities")
    print()

    # Step 2: Preprocess and engineer features
    print("Step 2: Feature engineering...")
    preprocessor = NIRFDataPreprocessor()
    df_features = preprocessor.create_features(df)
    X, y, df_clean = preprocessor.prepare_training_data(df_features)
    print(f"  Created {len(preprocessor.feature_columns)} features")
    print()

    # Step 3: Split data
    print("Step 3: Splitting data...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"  Training set: {len(X_train)} samples")
    print(f"  Test set: {len(X_test)} samples")
    print()

    # Step 4: Train ensemble model
    print("Step 4: Training models...")
    model = NIRFEnsembleModel()
    model.train(X_train, y_train, X_test, y_test)
    print()

    # Step 5: Evaluate model
    print("Step 5: Evaluating model performance...")
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)

    print(f"  Training R² Score: {train_r2:.4f}")
    print(f"  Test R² Score: {test_r2:.4f}")
    print(f"  Test RMSE: {test_rmse:.4f}")
    print(f"  Test MAE: {test_mae:.4f}")
    print()

    # Step 6: Make predictions with confidence intervals
    print("Step 6: Generating predictions with confidence intervals...")
    pred, lower, upper = model.predict_with_confidence(X_test)

    results_df = pd.DataFrame({
        'Actual_Score': y_test.values,
        'Predicted_Score': pred,
        'Lower_Bound': lower,
        'Upper_Bound': upper,
        'Error': np.abs(y_test.values - pred)
    })

    print(results_df.head(10))
    print()

    # Step 7: Time series forecasting
    print("Step 7: Forecasting future metrics...")
    forecaster = TimeSeriesForecaster()
    target_university = df['University'].iloc[0]

    metrics_to_forecast = ['Publications_Scopus', 'Citations_Total',
                          'Patents_Granted', 'Funding_Amount_Lakhs']

    forecasts = forecaster.forecast_all_metrics(
        df, target_university, metrics_to_forecast, periods=3
    )

    print(f"  Forecasts for {target_university}:")
    for metric, values in forecasts.items():
        print(f"    {metric}: {values}")
    print()

    # Step 8: Create dashboard
    print("Step 8: Generating strategic dashboard...")
    dashboard = StrategicDashboard(model, preprocessor, df)

    # Visualizations
    print("\n  Generating feature importance plot...")
    dashboard.plot_feature_importance(top_n=15)

    print("\n  Generating prediction accuracy plot...")
    dashboard.plot_prediction_accuracy(y_test, y_pred_test)

    print("\n  Generating rank trajectory plot...")
    # Simulate rank forecasts (simplified)
    historical_ranks = df[df['University'] == target_university]['NIRF_Rank'].tolist()
    forecasted_ranks = [historical_ranks[-1] - i for i in range(1, 4)]  # Improving trend
    dashboard.plot_rank_trajectory(target_university, forecasted_ranks)

    # Step 9: Strategic recommendations
    current_metrics = df[df['University'] == target_university].iloc[-1].to_dict()
    dashboard.generate_strategic_recommendations(target_university, current_metrics)

    print("="*70)
    print("PIPELINE EXECUTION COMPLETE")
    print("="*70)

    return model, preprocessor, dashboard, df, results_df

# ============================================================================
# EXECUTE THE PIPELINE
# ============================================================================

if __name__ == "__main__":
    model, preprocessor, dashboard, data, results = main_pipeline()

    # Save the model (optional)
    print("\n💾 To save the model, uncomment the following lines:")
    print("# import joblib")
    print("# joblib.dump(model, 'nirf_ensemble_model.pkl')")
    print("# joblib.dump(preprocessor, 'nirf_preprocessor.pkl')")

RANKOPT STRATEGIC FORESIGHT PLATFORM
NIRF Predictive Engine v1.0

Step 1: Loading data...
  Loaded data: 150 records, 30 universities

Step 2: Feature engineering...
  Created 31 features

Step 3: Splitting data...
  Training set: 120 samples
  Test set: 30 samples

Step 4: Training models...
Training Ensemble Models...

Training xgboost...
  CV R² Score: 0.7918 (+/- 0.1054)

Training lightgbm...
  CV R² Score: 0.7494 (+/- 0.1092)

Training random_forest...
  CV R² Score: 0.8172 (+/- 0.0972)

Training Complete!

Step 5: Evaluating model performance...
  Training R² Score: 0.9878
  Test R² Score: 0.7968
  Test RMSE: 2.8599
  Test MAE: 1.8933

Step 6: Generating predictions with confidence intervals...
   Actual_Score  Predicted_Score  Lower_Bound  Upper_Bound     Error
0     25.819409        25.733032    23.125442    28.340623  0.086376
1     46.902744        40.159867    39.127433    41.192301  6.742877
2     42.014894        35.760395    34.843373    36.677418  6.254498
3     27.82471


  Generating prediction accuracy plot...



  Generating rank trajectory plot...



STRATEGIC RECOMMENDATIONS FOR University_1
Target Rank: 30

1. 👨‍🏫 Enhance faculty quality (Importance: 23.606)
   → Recruit PhD-qualified faculty and improve ratio

2. 💰 Increase research funding (Importance: 18.002)
   → Apply for more government and industry grants

3. 👨‍🏫 Enhance faculty quality (Importance: 13.215)
   → Recruit PhD-qualified faculty and improve ratio

4. 📚 Increase research publications (Importance: 12.805)
   → Incentivize faculty publications in Q1/Q2 journals

5. 📊 Improve citation impact (Importance: 11.209)
   → Focus on high-impact research areas and collaborations


PIPELINE EXECUTION COMPLETE

💾 To save the model, uncomment the following lines:
# import joblib
# joblib.dump(model, 'nirf_ensemble_model.pkl')
# joblib.dump(preprocessor, 'nirf_preprocessor.pkl')
